# **Loan Default Prediction**

## **Problem Definition**

### **The Context:**

 Banks rely heavily on profits from home loans, but defaults (NPAs) can seriously reduce profitability. Current approval processes are manual, time-consuming, and prone to human error and bias. With the rise of data science, there is an opportunity to build models that improve efficiency, fairness, and compliance while reducing risk.

### **The objective:**

 The main objective is to develop a classification model to predict the likelihood of loan default for new applicants. The model should:

* Support the bank in approving/rejecting loan applications more consistently and fairly.

* Highlight the most important factors that drive default risk, providing actionable insights for loan officers.

* Balance predictive accuracy with interpretability, ensuring transparency in decision-making. 

### **The key questions:**

* Prediction – Can we accurately predict whether a loan applicant is likely to default?

* Feature Importance – What applicant characteristics (income, employment status, existing debt, credit history, etc.) are most predictive of default?

* Bias & Fairness – How do we ensure that the model does not replicate or amplify human biases present in past approvals?

* Interpretability – Can the model’s predictions be explained in a way that satisfies regulatory requirements (e.g., justification for rejections)?

* Business Value – To what extent can this model reduce losses from NPAs and improve the efficiency of the credit approval process?

### **The problem formulation**:

We need a data-driven credit scoring model that predicts the likelihood of default, supports transparent and fair loan decisions, and helps banks minimize losses from bad loans.



## **Data Description:**
The Home Equity dataset (HMEQ) contains baseline and loan performance information for 5,960 recent home equity loans. The target (BAD) is a binary variable that indicates whether an applicant has ultimately defaulted or has been severely delinquent. This adverse outcome occurred in 1,189 cases (20 percent). 12 input variables were registered for each applicant.


* **BAD:** 1 = Client defaulted on loan, 0 = loan repaid

* **LOAN:** Amount of loan approved.

* **MORTDUE:** Amount due on the existing mortgage.

* **VALUE:** Current value of the property. 

* **REASON:** Reason for the loan request. (HomeImp = home improvement, DebtCon= debt consolidation which means taking out a new loan to pay off other liabilities and consumer debts) 

* **JOB:** The type of job that loan applicant has such as manager, self, etc.

* **YOJ:** Years at present job.

* **DEROG:** Number of major derogatory reports (which indicates a serious delinquency or late payments). 

* **DELINQ:** Number of delinquent credit lines (a line of credit becomes delinquent when a borrower does not make the minimum required payments 30 to 60 days past the day on which the payments were due). 

* **CLAGE:** Age of the oldest credit line in months. 

* **NINQ:** Number of recent credit inquiries. 

* **CLNO:** Number of existing credit lines.

* **DEBTINC:** Debt-to-income ratio (all your monthly debt payments divided by your gross monthly income. This number is one way lenders measure your ability to manage the monthly payments to repay the money you plan to borrow.

## **Import Libraries and Load Data**

This section loads the raw HMEQ dataset and keeps categorical variables in their original form for descriptive analysis. Dummy encoding is deferred until the modelling/pre-processing stage.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.options.display.float_format = "{:,.2f}".format
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", None)

sns.set_theme(style="whitegrid", context="notebook")
PROJECT_DIR = Path.cwd()
DATA_PATH = PROJECT_DIR / "hmeq.csv"

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Could not find hmeq.csv at {DATA_PATH}")

df_raw = pd.read_csv(DATA_PATH)
df = df_raw.copy()


## **Data Overview**

The first checks confirm the size, structure, field names, missingness, duplicates, and target balance before any transformations are applied.


In [ ]:
print(f"Dataset shape: {df.shape[0]:,} rows x {df.shape[1]:,} columns")
display(df.head())


In [ ]:
column_overview = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "non_null_count": df.notna().sum().values,
    "missing_count": df.isna().sum().values,
    "missing_%": df.isna().mean().mul(100).round(2).values,
})

display(column_overview)
print("Column names:", list(df.columns))
print(f"Duplicate rows: {df.duplicated().sum():,}")


In [ ]:
target_balance = (
    df["BAD"]
    .value_counts(dropna=False)
    .rename_axis("BAD")
    .reset_index(name="count")
)
target_balance["percent"] = (target_balance["count"] / len(df) * 100).round(2)
target_balance["class_label"] = target_balance["BAD"].map({0: "Repaid", 1: "Defaulted / severely delinquent"})
display(target_balance[["BAD", "class_label", "count", "percent"]])


**Comment.** The dataset contains 5,960 loan records and 13 variables. `BAD` is the target indicator: `BAD = 1` means the client defaulted or was severely delinquent, while `BAD = 0` means the loan was repaid. The classes are imbalanced, with defaults representing about one fifth of the observations.


## **Data Dictionary Alignment**

The variable descriptions below follow the project problem statement. They are used to keep the analysis aligned with the business meaning of each field.


In [ ]:
data_dictionary = pd.DataFrame({
    "Variable": [
        "BAD", "LOAN", "MORTDUE", "VALUE", "REASON", "JOB", "YOJ", "DEROG",
        "DELINQ", "CLAGE", "NINQ", "CLNO", "DEBTINC"
    ],
    "Description": [
        "Default indicator: 1 = defaulted or severely delinquent; 0 = repaid",
        "Amount of loan approved",
        "Amount due on existing mortgage",
        "Current property value",
        "Reason for loan request",
        "Applicant job type",
        "Years at present job",
        "Number of major derogatory reports",
        "Number of delinquent credit lines",
        "Age of oldest credit line in months",
        "Number of recent credit inquiries",
        "Number of existing credit lines",
        "Debt-to-income ratio",
    ],
})

display(data_dictionary)


## **Variable Groups**

`BAD` is analysed as the target/class variable rather than as an ordinary continuous numerical feature.


In [ ]:
target = "BAD"
categorical_predictors = ["REASON", "JOB"]
numerical_predictors = [col for col in df.columns if col not in categorical_predictors + [target]]

print("Target variable:", target)
print("Categorical predictors:", categorical_predictors)
print("Numerical predictors:", numerical_predictors)


## **Missing-Value Analysis**

Missing values are inspected before imputation. Imputation choices are part of the modelling/pre-processing stage and should be reviewed separately.


In [ ]:
missing_summary = (
    df.isna()
    .agg(["sum", "mean"])
    .T
    .rename(columns={"sum": "missing_count", "mean": "missing_%"})
)
missing_summary["missing_%"] = missing_summary["missing_%"].mul(100).round(2)
missing_summary = missing_summary.sort_values("missing_%", ascending=False)

display(missing_summary)


**Comment.** `DEBTINC`, `DEROG`, `DELINQ`, `MORTDUE`, `YOJ`, `NINQ`, `CLAGE`, `JOB`, `REASON`, `VALUE`, and `CLNO` contain missing values. `DEBTINC` has the highest missingness and should receive special attention during modelling because it is also likely to be credit-risk relevant. Missingness indicators may be useful later for variables where the absence of information itself could be informative.


## **Target Distribution**


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
plot_data = target_balance.copy()
plot_data["BAD"] = plot_data["BAD"].astype(str)
sns.barplot(data=plot_data, x="BAD", y="count", ax=ax, color="#4C78A8")
ax.set_title("Distribution of Loan Outcomes")
ax.set_xlabel("BAD (0 = repaid, 1 = defaulted/severely delinquent)")
ax.set_ylabel("Number of clients")

for container in ax.containers:
    ax.bar_label(container, labels=[f"{p:.1f}%" for p in plot_data["percent"]], padding=3)

plt.tight_layout()
plt.show()


**Comment.** Approximately 20% of clients defaulted or became severely delinquent. This imbalance should be considered when evaluating future classification models, especially for recall and false-negative costs.


## **Summary Statistics for Numerical Predictors**


In [ ]:
numerical_summary = df[numerical_predictors].describe().T
numerical_summary["missing_%"] = df[numerical_predictors].isna().mean().mul(100).round(2)
numerical_summary["skew"] = df[numerical_predictors].skew(numeric_only=True).round(2)

display(numerical_summary)


**Comment.** The monetary variables (`LOAN`, `MORTDUE`, and `VALUE`) operate on a much larger scale than the credit-count variables. Several variables appear right-skewed, especially credit-history counts such as `DEROG`, `DELINQ`, and `NINQ`, where many clients have zero events and a smaller number have high counts. `DEBTINC` also appears important to inspect because it combines substantial missingness with a potentially wide range.


## **Categorical Predictor Summaries**

The raw categorical variables are analysed before dummy encoding.


In [ ]:
def categorical_summary_table(data, column):
    counts = data[column].fillna("Missing").value_counts(dropna=False)
    summary = counts.rename_axis(column).reset_index(name="count")
    summary["percent"] = (summary["count"] / len(data) * 100).round(2)
    return summary

for column in categorical_predictors:
    print(f"\n{column}")
    display(categorical_summary_table(df, column))


**Comment.** `REASON` is dominated by debt-consolidation loans, with home-improvement loans forming the smaller named category. `JOB` is more dispersed; the largest groups are office, professional/executive, and other job types. Rare categories should be reviewed during modelling because they can produce unstable category-level estimates.


## **Numerical Distributions and Outliers**


In [ ]:
def plot_hist_box_grid(data, columns, bins=30, cols=2):
    rows = int(np.ceil(len(columns) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 4.5 * rows))
    axes = np.array(axes).reshape(rows, cols)

    for ax, column in zip(axes.ravel(), columns):
        sns.histplot(data=data, x=column, bins=bins, kde=True, ax=ax, color="#4C78A8")
        ax.axvline(data[column].median(), color="#222222", linestyle="--", linewidth=1, label="Median")
        ax.set_title(f"Distribution of {column}")
        ax.set_xlabel(column)
        ax.set_ylabel("Count")
        ax.legend()

    for ax in axes.ravel()[len(columns):]:
        ax.set_visible(False)

    plt.tight_layout()
    plt.show()

key_numeric = ["LOAN", "MORTDUE", "VALUE", "DEBTINC", "YOJ", "CLAGE"]
plot_hist_box_grid(df, key_numeric, bins=35, cols=3)


In [ ]:
def plot_box_grid(data, columns, cols=3):
    rows = int(np.ceil(len(columns) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 3.5 * rows))
    axes = np.array(axes).reshape(rows, cols)

    for ax, column in zip(axes.ravel(), columns):
        sns.boxplot(data=data, x=column, ax=ax, color="#72B7B2")
        ax.set_title(f"Boxplot of {column}")
        ax.set_xlabel(column)

    for ax in axes.ravel()[len(columns):]:
        ax.set_visible(False)

    plt.tight_layout()
    plt.show()

plot_box_grid(df, key_numeric, cols=3)


**Comment.** The distributions suggest right-skewness and visible high-end outliers in the monetary variables and `DEBTINC`. These observations do not necessarily indicate data errors; large loans, high property values, or high debt-to-income ratios can be genuine. Future modelling should consider robust preprocessing or tree-based models that are less sensitive to scale and skewness.


## **Categorical Distributions**


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for ax, column in zip(axes, categorical_predictors):
    cat_data = categorical_summary_table(df, column)
    sns.barplot(data=cat_data, x=column, y="count", ax=ax, color="#4C78A8")
    ax.set_title(f"Distribution of {column}")
    ax.set_xlabel(column)
    ax.set_ylabel("Number of clients")
    ax.tick_params(axis="x", rotation=30)
    for container in ax.containers:
        ax.bar_label(container, labels=[f"{p:.1f}%" for p in cat_data["percent"]], padding=3, fontsize=9)

plt.tight_layout()
plt.show()


## **Numerical Predictors by Default Status**


In [ ]:
def plot_by_target_box_grid(data, columns, target="BAD", cols=3):
    rows = int(np.ceil(len(columns) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 4 * rows))
    axes = np.array(axes).reshape(rows, cols)

    for ax, column in zip(axes.ravel(), columns):
        sns.boxplot(data=data, x=target, y=column, ax=ax, palette="Set2", hue = target, legend = False)
        ax.set_title(f"{column} by default status")
        ax.set_xlabel("BAD (0 = repaid, 1 = defaulted)")
        ax.set_ylabel(column)

    for ax in axes.ravel()[len(columns):]:
        ax.set_visible(False)

    plt.tight_layout()
    plt.show()

risk_numeric = ["LOAN", "MORTDUE", "VALUE", "DEBTINC", "DEROG", "DELINQ", "NINQ", "CLNO", "CLAGE"]
plot_by_target_box_grid(df, risk_numeric, target=target, cols=3)


In [ ]:
def plot_target_hist_grid(data, columns, target="BAD", bins=30, cols=3):
    rows = int(np.ceil(len(columns) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 4 * rows))
    axes = np.array(axes).reshape(rows, cols)

    plot_data = data.copy()
    plot_data[target] = plot_data[target].map({0: "Repaid", 1: "Defaulted"})

    for ax, column in zip(axes.ravel(), columns):
        sns.histplot(
            data=plot_data,
            x=column,
            hue=target,
            bins=bins,
            stat="density",
            common_norm=False,
            element="step",
            ax=ax,
        )
        ax.set_title(f"{column} distribution by loan outcome")
        ax.set_xlabel(column)
        ax.set_ylabel("Density")

    for ax in axes.ravel()[len(columns):]:
        ax.set_visible(False)

    plt.tight_layout()
    plt.show()

plot_target_hist_grid(df, ["DEBTINC", "DEROG", "DELINQ", "NINQ", "CLAGE", "YOJ"], target=target, bins=30, cols=3)


**Comment.** Defaults appear more common among clients with weaker credit-history measures, particularly higher `DEROG`, `DELINQ`, `NINQ`, and `DEBTINC`. The relationship should be described as association rather than causation because this is observational data.


## **Default Rates by Categorical Predictors**


In [ ]:
def default_rate_table(data, column, target="BAD"):
    temp = data[[column, target]].copy()
    temp[column] = temp[column].fillna("Missing")
    summary = (
        temp.groupby(column, dropna=False)[target]
        .agg(count="size", default_rate="mean")
        .reset_index()
    )
    summary["default_rate_%"] = summary["default_rate"].mul(100).round(2)
    return summary.sort_values("default_rate_%", ascending=False)

for column in categorical_predictors:
    print(f"\nDefault rate by {column}")
    display(default_rate_table(df, column, target=target))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for ax, column in zip(axes, categorical_predictors):
    rate_data = default_rate_table(df, column, target=target)
    sns.barplot(data=rate_data, x=column, y="default_rate_%", ax=ax, color="#F58518")
    ax.set_title(f"Default rate by {column}")
    ax.set_xlabel(column)
    ax.set_ylabel("Default rate (%)")
    ax.tick_params(axis="x", rotation=30)
    for container in ax.containers:
        ax.bar_label(container, fmt="%.1f", padding=3, fontsize=9)

plt.tight_layout()
plt.show()


**Comment.** Default rates vary across `REASON` and `JOB`, suggesting that categorical information may add predictive value. However, small or missing categories should be interpreted cautiously because their rates may be less stable.


## **Correlation Among Numerical Predictors**


In [ ]:
corr_cols = numerical_predictors
corr = df[corr_cols].corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, cmap="vlag", center=0, annot=True, fmt=".2f", linewidths=0.5, ax=ax)
ax.set_title("Correlation Heatmap for Numerical Predictors")
plt.tight_layout()
plt.show()


In [ ]:
target_correlations = (
    df[numerical_predictors + [target]]
    .corr(numeric_only=True)[target]
    .drop(target)
    .to_frame("correlation_with_BAD")
)
target_correlations["abs_correlation"] = target_correlations["correlation_with_BAD"].abs()
target_correlations = (
    target_correlations
    .sort_values("abs_correlation", ascending=False)
    .drop(columns="abs_correlation")
)

display(target_correlations)


**Comment.** `MORTDUE` and `VALUE` are positively correlated, as expected for mortgage and property-value measures. Credit-history variables such as `DELINQ`, `DEROG`, `NINQ`, and `DEBTINC` show the clearest associations with `BAD`, which is consistent with their role as credit-risk indicators. Correlations are descriptive and do not establish causal effects.


## **EDA Summary**

The raw EDA suggests several modelling-relevant patterns:

- About one fifth of clients defaulted or were severely delinquent, so class imbalance should be handled carefully during model evaluation.
- `DEBTINC` has substantial missingness and appears associated with default; its missingness and imputation strategy need careful treatment later.
- Monetary variables and several credit-count variables are right-skewed and contain high-end outliers that may be genuine business observations.
- `DEROG`, `DELINQ`, `NINQ`, and `DEBTINC` appear related to default status and are likely important predictors.
- Default rates differ across `REASON` and `JOB`, so the original categorical variables should be retained until the modelling pre-processing step.


## **Modelling Pre-Processing Placeholder**

The exploratory analysis above used the raw dataset. The following temporary modelling copy reproduces the earlier notebook's basic missing-value treatment and dummy encoding so that the existing modelling cells can still run. These choices should be reviewed and improved in the modelling revision pass.


In [ ]:
df_model = df_raw.copy()

# Keep track of missingness for variables where absence of information may be predictive.
for column in ["VALUE", "MORTDUE", "DEBTINC"]:
    df_model[f"{column}_missing"] = df_model[column].isna().astype(int)

# Basic numeric imputation retained for compatibility with the existing modelling section.
median_impute_cols = ["VALUE", "MORTDUE", "DEROG", "DELINQ", "NINQ", "CLAGE", "DEBTINC"]
for column in median_impute_cols:
    df_model[column] = df_model[column].fillna(df_model[column].median())

zero_impute_cols = ["CLNO", "YOJ"]
for column in zero_impute_cols:
    df_model[column] = df_model[column].fillna(0)

# Categorical encoding begins only after EDA.
df_model[categorical_predictors] = df_model[categorical_predictors].fillna("Missing")
df = pd.get_dummies(df_model, columns=categorical_predictors, drop_first=True)

print(f"Prepared modelling DataFrame shape: {df.shape[0]:,} rows x {df.shape[1]:,} columns")
display(df.head())


## **Model Building - Approach**
- Data preparation
- Partition the data into train and test set
- Build the model
- Fit on the train data
- Tune the model
- Test the model on test set

### Logistic Regression

In [ ]:
#Importing further required libraries

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, cross_val_predict, GridSearchCV
from sklearn.pipeline import Pipeline

We now run two logistic regression models, a non-regularized and a regularized (Lasso) one. 

In [ ]:
# Train/test split ---
X = df.drop(columns=["BAD"])
y = df["BAD"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Define the two logistic models (regularized and non) 
# (a) Plain logistic regression (no regularization)
pipe_plain = Pipeline([
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(
        penalty=None,          
        solver="lbfgs",
        max_iter=5000,
        class_weight="balanced"
    ))
])

# (b) L1-penalized logistic regression (Lasso)
pipe_l1 = Pipeline([
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(
        penalty="l1",
        solver="saga",
        max_iter=5000,
        class_weight="balanced"
    ))
])

# Hyperparameter grid for C (inverse regularization strength)
param_grid = {"logreg__C": np.logspace(-3, 2, 10)}
grid_l1 = GridSearchCV(pipe_l1, param_grid, cv=cv, scoring="roc_auc", n_jobs=-1, refit=True)

# Fit both models 
grid_l1.fit(X_train, y_train)
pipe_plain.fit(X_train, y_train)

# Compute metrics on TEST
def eval_metrics(name, est, X_te, y_te, cv_auc=None):
    y_hat = est.predict(X_te)
    y_proba = est.predict_proba(X_te)[:, 1]
    tn, fp, fn, tp = confusion_matrix(y_te, y_hat).ravel()
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_te, y_hat),
        "Precision": precision_score(y_te, y_hat, zero_division=0),
        "Recall": recall_score(y_te, y_hat, zero_division=0),
        "F1": f1_score(y_te, y_hat, zero_division=0),
        "ROC_AUC": roc_auc_score(y_te, y_proba),
        "CV_ROC_AUC": cv_auc
    }

# Cross-validation AUCs 
plain_cv_auc = cross_val_score(pipe_plain, X_train, y_train, cv=cv, scoring="roc_auc").mean()
l1_cv_auc = grid_l1.best_score_

# Collect results in a tidy DataFrame
results_df = pd.DataFrame([
    eval_metrics("LogReg (no penalty)", pipe_plain, X_test, y_test, cv_auc=plain_cv_auc),
    eval_metrics(f"LogReg L1 (C={grid_l1.best_params_['logreg__C']:.4g})", grid_l1.best_estimator_, X_test, y_test, cv_auc=l1_cv_auc)
]).set_index("Model")

# Round numeric columns for readability
cols_to_round = ["Accuracy","Precision","Recall","F1","ROC_AUC","CV_ROC_AUC"]
results_df[cols_to_round] = results_df[cols_to_round].round(3)

print(results_df)


**Interpretation:**

- The two models perform virtually identically on all metrics.

- ROC-AUC is about 0.89–0.90 so strong for credit scoring (excellent discriminatory power).

- The L1 (lasso) model did not improve performance. This could be a sign that the predictors are not excessively noisy or collinear.

- Recall = 0.80 means the model catches 80% of actual defaulters; precision = 0.58 means 42% of predicted defaulters are false alarms.

**Conclusion:** I will consider the non regularized Logistic Regression model as coefficients are more interpretable and the regularized model does not improve performance.

In [ ]:
results_log = results_df.loc[["LogReg (no penalty)"]].copy()

print(results_log)

In [ ]:
# Coefficients from L2 model 
coef_l2 = pd.Series(pipe_plain.named_steps["logreg"].coef_[0], index=X.columns)

# Coefficients from best L1 (lasso) model
best_l1 = grid_l1.best_estimator_
coef_l1 = pd.Series(best_l1.named_steps["logreg"].coef_[0], index=X.columns)

# Combine into one dataframe for easy comparison
coef_compare = pd.DataFrame({
    "L2_logit": coef_l2,
    "L1_logit": coef_l1
}).sort_values(by="L2_logit", ascending=False)

# Add an indicator for features zeroed out by Lasso
coef_compare["L1_zeroed"] = coef_compare["L1_logit"].abs() < 1e-6

# Display top and bottom features
print(coef_compare.head(10))
print(coef_compare.tail(10))


Comment:

* Comparing L1- and L2-regularized logistic regressions reveals that Lasso retains 100 % of the original features, setting no coefficients to zero. The main drivers of default remain consistent across both models, including high DEBTINC_missing,DELINQ, DEBTINC, VALUE_missing, DEROG, as strong positive predictors, while higher CLAGE and CLNO are negatively associated with default probability.
* Both models rank in the same way the features and coefficients are very close. 

In [ ]:
!pip install statsmodels

In [ ]:
pip install --upgrade statsmodels

I want to inspect the coefficients of the Logistic Regression (Lasso) model in more detail. 

In [ ]:

import statsmodels.api as sm
from sklearn.utils.class_weight import compute_sample_weight


# --- pull the trained pieces from your plain (non-regularized) pipeline ---
scaler_plain = pipe_plain.named_steps["scaler"]
# NOTE: sklearn's LogisticRegression doesn't provide SE/p-values, so we refit in statsmodels

# --- scale X using the already-fitted scaler (so results align with the pipeline) ---
X_train_scaled = pd.DataFrame(
    scaler_plain.transform(X_train),
    columns=X.columns,
    index=X_train.index
)
X_test_scaled = pd.DataFrame(
    scaler_plain.transform(X_test),
    columns=X.columns,
    index=X_test.index
)

# --- add intercept ---
X_train_sm = sm.add_constant(X_train_scaled, has_constant="add")
X_test_sm  = sm.add_constant(X_test_scaled,  has_constant="add")

# --- replicate class_weight="balanced" from sklearn ---
# sklearn uses: weight_c = n_samples / (n_classes * n_samples_c)
sample_w = compute_sample_weight(class_weight="balanced", y=y_train)

# --- fit GLM Binomial (supports weights); robust SEs via cov_type="HC1" ---
glm_binom = sm.GLM(y_train, X_train_sm, family=sm.families.Binomial(), freq_weights=sample_w)
res = glm_binom.fit(cov_type="HC1")  # HC1 robust covariance like in your code
print(res.summary())

# --- tidy table with odds ratios and 95% CI ---
ci = res.conf_int(alpha=0.05)
coef_table = pd.DataFrame({
    "coef":      res.params,
    "se_robust": res.bse,
    "p_value":   res.pvalues,
})
coef_table[["ci2.5%","ci97.5%"]] = ci
coef_table["odds_ratio"] = np.exp(coef_table["coef"])
coef_table["OR_ci2.5%"]  = np.exp(coef_table["ci2.5%"])
coef_table["OR_ci97.5%"] = np.exp(coef_table["ci97.5%"])
coef_table = coef_table.sort_values("odds_ratio", ascending=False)
print("\nCoef table (robust):\n", coef_table)

# --- evaluate on TEST to sanity-check alignment with the sklearn pipeline ---
p_test = res.predict(X_test_sm)
y_pred = (p_test >= 0.5).astype(int)

print("\nTest ROC AUC (statsmodels GLM):", roc_auc_score(y_test, p_test))
print("Test Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Test Classification Report:\n", classification_report(y_test, y_pred, zero_division=0))



### **Interpretation of Logistic Regression Coefficients (Ranked by Odds Ratio — Significant Predictors Only):**

| Rank | Variable | Sign | Odds Ratio | 95% CI (OR) | Economic Interpretation |
|------|-----------|------|-------------|--------------|--------------------------|
| **1** | **DEBTINC_missing** | **+** | **2.95** | [2.73 – 3.20] | Borrowers missing debt-to-income info are ≈ 3× more likely to default. Missingness is informative—often linked to hidden debt or unreliable data. |
| **2** | **DELINQ** | **+** | **2.25** | [2.00 – 2.53] | Each additional past delinquency more than doubles default odds; strong repayment-history effect. |
| **3** | **DEBTINC** | **+** | **1.86** | [1.64 – 2.10] | Higher debt-to-income ratio greatly increases risk; key measure of repayment capacity. |
| **4** | **VALUE_missing** | **+** | **1.81** | [1.53 – 2.14] | Missing property value nearly doubles default risk — indicative of incomplete or low-quality applications. |
| **5** | **DEROG** | **+** | **1.65** | [1.45 – 1.87] | Presence of derogatory credit records increases odds by ~65 %. |
| **6** | **JOB_Other** | **+** | **1.25** | [1.11 – 1.41] | “Other” job category borrowers have ≈ 25 % higher odds — heterogeneous, possibly unstable occupations. |
| **7** | **JOB_Sales** | **+** | **1.24** | [1.13 – 1.36] | Sales occupations entail income volatility; ~24 % higher default odds. |
| **8** | **NINQ** | **+** | **1.23** | [1.13 – 1.34] | More recent credit inquiries raise risk (+23 %); signals active credit-seeking or liquidity strain. |
| **9** | **VALUE** | **+** | **1.18** | [1.09 – 1.29] | Higher property value slightly raises odds, possibly reflecting over-leveraging conditional on income. |
| **10** | **JOB_Self** | **+** | **1.18** | [1.08 – 1.28] | Self-employed borrowers are ~18 % more likely to default, reflecting income volatility. |
| **11** | **REASON_HomeImp** | **+** | **1.09** | [1.00 – 1.20] | Home-improvement loans marginally riskier than debt consolidation; this is a bit counterintuitive. |
| **12** | **JOB_Office** | **–** | **0.85** | [0.76 – 0.95] | Office jobs carry ≈ 15 % lower default odds; steady salaried employment. |
| **13** | **MORTDUE** | **–** | **0.86** | [0.76 – 0.96] | Higher mortgage balances associate with lower risk — wealthier, better-screened borrowers. |
| **14** | **LOAN** | **–** | **0.84** | [0.76 – 0.93] | Larger loan size predicts lower default — consistent with lender screening of safer clients. |
| **15** | **CLNO** | **–** | **0.78** | [0.71 – 0.86] | More open credit lines reduce risk — experienced and diversified borrowers. |
| **16** | **CLAGE** | **–** | **0.68** | [0.57 – 0.80] | Longer credit history sharply reduces default odds (–32 % per SD), capturing borrower maturity. |

### Summary Insights:
- **Risk-increasing factors:** Missing financial data, poor credit history, and high debt burden dominate.
- **Risk-reducing factors:** Long credit history, more credit lines, stable office employment, and larger loans.
- The model emphasizes **behavioural transparency** and **financial capacity** as the main drivers of default probability.


### Decision Tree

In [ ]:

# Baseline Decision Tree (no tuning)

dt_clf = DecisionTreeClassifier(
    criterion="gini",         
    class_weight="balanced",  
    random_state=42
)

#  Fit the model
dt_clf.fit(X_train, y_train)

# Predict 
y_pred  = dt_clf.predict(X_test)
y_proba = dt_clf.predict_proba(X_test)[:, 1]

# Evaluate 
results = [{
    "Model":      "Decision Tree (baseline)",
    "Accuracy":   accuracy_score(y_test, y_pred),
    "Precision":  precision_score(y_test, y_pred, zero_division=0),
    "Recall":     recall_score(y_test, y_pred, zero_division=0),
    "F1":         f1_score(y_test, y_pred, zero_division=0),
    "ROC-AUC":    roc_auc_score(y_test, y_proba)
}]

results_dt = pd.DataFrame(results)
print("\nTest-set performance:")
print(results_dt.round(3))

print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred))

# Feature importance 
fi = pd.Series(dt_clf.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print("\nTop 10 features:")
print(fi.head(10))

Very poor performance for this baseline decision tree. Especially, if we focus attention on Precision, Recall, or ROC-AUC, the model performs worse than our Logistic Regression models. 

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree


# We visualize the tree

dt_classifier_visualize = DecisionTreeClassifier(
    random_state=42,
    max_depth=3,      
    class_weight="balanced",
    criterion="gini"
)

# Fit on training data 
dt_classifier_visualize.fit(X_train, y_train)


plt.figure(figsize=(20, 10))
plot_tree(
    dt_classifier_visualize,
    feature_names=X_train.columns,
    class_names=["Repaid (0)", "Defaulted (1)"],
    filled=True,
    rounded=True,
    fontsize=10
)
plt.title("Decision Tree (max_depth=3)")
plt.show()


### **Decision Tree - Hyperparameter Tuning**

* Hyperparameter tuning is tricky in the sense that **there is no direct way to calculate how a change in the hyperparameter value will reduce the loss of your model**, so we usually resort to experimentation. We'll use Grid search to perform hyperparameter tuning.
* **Grid search is a tuning technique that attempts to compute the optimum values of hyperparameters.** 
* **It is an exhaustive search** that is performed on the specific parameter values of a model.
* The parameters of the estimator/model used to apply these methods are **optimized by cross-validated grid-search** over a parameter grid.

**Criterion {“gini”, “entropy”}**

The function to measure the quality of a split. Supported criteria are “gini” for the Gini impurity and “entropy” for the information gain.

**max_depth** 

The maximum depth of the tree. If None, then nodes are expanded until all leaves are pure or until all leaves contain less than min_samples_split samples.

**min_samples_leaf**

The minimum number of samples is required to be at a leaf node. A split point at any depth will only be considered if it leaves at least min_samples_leaf training samples in each of the left and right branches. This may have the effect of smoothing the model, especially in regression.

You can learn about more Hyperpapameters on this link and try to tune them. 

https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html


In [ ]:

# Cross-validated Decision Tree

param_grid = {
    "max_depth": [3, 5, 7, 9, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 3, 5],
    "criterion": ["gini", "entropy"],
    "class_weight": ["balanced"]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid=param_grid,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1
)
grid.fit(X_train, y_train)

best_dt = grid.best_estimator_
print("Best parameters:", grid.best_params_)
print(f"Mean CV ROC-AUC: {grid.best_score_:.3f}")


# Evaluate on Test Set

y_pred  = best_dt.predict(X_test)
y_proba = best_dt.predict_proba(X_test)[:, 1]

# Collect metrics
results = [{
    "Model":      "Decision Tree (CV-tuned)",
    "Accuracy":   accuracy_score(y_test, y_pred),
    "Precision":  precision_score(y_test, y_pred, zero_division=0),
    "Recall":     recall_score(y_test, y_pred, zero_division=0),
    "F1":         f1_score(y_test, y_pred, zero_division=0),
    "ROC-AUC":    roc_auc_score(y_test, y_proba)
}]

results_dt_best = pd.DataFrame(results)
print("\nTest-set performance:")
print(results_dt_best.round(3))

print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred))


# Feature Importances

fi = pd.Series(best_dt.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print("\nTop 10 features:")
print(fi.head(10))

In [ ]:
print(pd.concat(
    [results_dt, results_dt_best],
    axis=0
))


In [ ]:
dt_clf.get_params()

**Comment:**

* After the tuning, the Decision tree model performs better in some of the metrics (recall and ROC_AUC).
* The fine tuning picked max_depth': 5, 'min_samples_leaf': 5, 'min_samples_split': 2, so the difference with the baseline model is the criterion, the depth, and the min_sample_leaf. 
* Even in the tuned Decision Tree model, the top features seem to overlap with those identified as most important by the Logistic Reg models.   

In [ ]:
from sklearn import tree

# visualize the tree-structure for this tuned Decision tree

print(tree.export_text(best_dt, feature_names=X_train.columns.tolist(), show_weights=True))

**Comment:**

* The tree is quite deep and granular, which suggests it’s capturing non-linear interactions among key credit variables.
* The first split (the root) is on DEBTINC_missing, meaning that whether the debt-to-income ratio is missing has strong predictive power — likely because missing financial data is itself a red flag for creditworthiness.
After that, the tree splits primarily on DEBTINC, DELINQ, CLAGE, DEROG. These are same indicators of default risk identified by our Logistic Reg model. They are also quite economically plausible.
* The model detects a threshold effect around a debt-to-income ratio of 43 %, beyond which default probability jumps sharply.
* Missing debt-to-income information strongly correlates with bad loans — possibly due to incomplete applications or higher-risk profiles. This is true even when DELINQ=0 for example. 

Key predictive insights:

- DEBTINC / DEBTINC_missing	Most influential variable; high values or missing data signal strong default risk.
- DELINQ, DEROG	Indicators of poor credit behaviour; even small positive values increase default probability sharply.
- CLAGE	Longer credit histories (higher CLAGE) reduce risk; younger borrowers or those with new credit lines are riskier.
- MORTDUE / VALUE	Secondary importance; smaller or missing property values correlate with default among high-risk applicants.
- YOJ, CLNO	Stability variables; more years on the job and more credit lines (financial maturity) lower default risk.

However, the tree is fairly deep and may be overfitting to idiosyncrasies of the training data. I will try to use an ensemble (Random Forest / Gradient Boosting) to capture the same nonlinear structure more robustly.

### **Building a Random Forest Classifier**

**Random Forest is a bagging algorithm where the base models are Decision Trees.** Samples are taken from the training data and on each sample a decision tree makes a prediction. 

**The results from all the decision trees are combined together and the final prediction is made using voting or averaging.**

In [ ]:
from sklearn.ensemble import RandomForestClassifier



# Baseline Random Forest (no tuning)

rf_clf = RandomForestClassifier(
    n_estimators=300,          
    criterion="gini",
    max_depth=None,           
    min_samples_leaf=1,
    max_features="sqrt",     
    class_weight="balanced", 
    n_jobs=-1,
    random_state=42
)

# Fit 
rf_clf.fit(X_train, y_train)

# Predict 
y_pred  = rf_clf.predict(X_test)
y_proba = rf_clf.predict_proba(X_test)[:, 1]

# Evaluate (same metrics as before) 
results = [{
    "Model":      "Random Forest (baseline)",
    "Accuracy":   accuracy_score(y_test, y_pred),
    "Precision":  precision_score(y_test, y_pred, zero_division=0),
    "Recall":     recall_score(y_test, y_pred, zero_division=0),
    "F1":         f1_score(y_test, y_pred, zero_division=0),
    "ROC-AUC":    roc_auc_score(y_test, y_proba)
}]
results_rf = pd.DataFrame(results)
print("\nTest-set performance:")
print(results_rf.round(3))

print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred))

# Feature importance 
fi = pd.Series(rf_clf.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print("\nTop 15 features (Random Forest):")
print(fi.head(15))


### **Random Forest Classifier Hyperparameter Tuning**

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
)
from scipy.stats import randint
import pandas as pd

# Cross-validation setup
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Define base model
rf_base = RandomForestClassifier(
    random_state=42,
    n_jobs=-1,
    class_weight="balanced_subsample",
    bootstrap=True
)

# Define efficient randomized search space
param_dist = {
    "n_estimators": randint(150, 350),
    "max_depth": [None, 8, 12, 16],
    "min_samples_leaf": randint(1, 8),
    "max_features": ["sqrt", "log2", 0.5],
    "max_samples": [None, 0.7, 0.9],
}

# Randomized search (faster than GridSearchCV)
rf_search = RandomizedSearchCV(
    rf_base,
    param_distributions=param_dist,
    n_iter=30,                  
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1,
    random_state=42,
    verbose=1
)
rf_search.fit(X_train, y_train)

best_rf = rf_search.best_estimator_

print("✅ Best parameters (Random Forest):")
print(rf_search.best_params_)
print(f"Mean CV ROC-AUC: {rf_search.best_score_:.3f}")

# Evaluate tuned RF on the test set
y_pred = best_rf.predict(X_test)
y_proba = best_rf.predict_proba(X_test)[:, 1]

rf_results = [{
    "Model": "Random Forest (tuned-fast)",
    "Accuracy":  accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred, zero_division=0),
    "Recall":    recall_score(y_test, y_pred, zero_division=0),
    "F1":        f1_score(y_test, y_pred, zero_division=0),
    "ROC-AUC":   roc_auc_score(y_test, y_proba),
    "CV_ROC-AUC": rf_search.best_score_
}]

results_rf_best = pd.DataFrame(rf_results).round(3)
print("\n=== Test-set performance ===")
print(results_rf_best)

# Confusion matrix
print("\n=== Confusion matrix ===")
print(confusion_matrix(y_test, y_pred))

# Feature importances (top 15)
fi = pd.Series(best_rf.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print("\n=== Top 15 features (Random Forest, tuned-fast) ===")
print(fi.head(15))



In [ ]:
print(pd.concat(
    [results_rf, results_rf_best],
    axis=0
))

In [ ]:
params_to_check = ["max_depth", "max_features", "max_samples", "min_samples_leaf", "n_estimators"]
baseline_params = {p: rf_base.get_params()[p] for p in params_to_check}
print(baseline_params)

**Comment:** 
* The fine-tuned random forest seems to perform better in some metrics than the baseline (recall and F1), slightly worse in precision and almost identical in ROC_AUC.
* fine tuned one has higher n_estimators (better generalization) and smaller min_sample_leaf (more granular split).

In [ ]:
def evaluate_model(model, X_tr, y_tr, X_te, y_te, name):
    y_pred  = model.predict(X_te)
    # Some estimators expose predict_proba, others decision_function
    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X_te)[:, 1]
    else:
        # fall back to a squashed decision function if needed
        df = model.decision_function(X_te)
        y_proba = 1 / (1 + np.exp(-df))
    results_df = pd.DataFrame([{
        "Model": name,
        "Accuracy":  accuracy_score(y_te, y_pred),
        "Precision": precision_score(y_te, y_pred, zero_division=0),
        "Recall":    recall_score(y_te, y_pred, zero_division=0),
        "F1":        f1_score(y_te, y_pred, zero_division=0),
        "ROC-AUC":   roc_auc_score(y_te, y_proba)
    }]).round(3)

    print(results_df)
    print("\nConfusion matrix:\n", confusion_matrix(y_te, y_pred))

    # Return the DataFrame, not a list
    return results_df


In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

gb_baseline = GradientBoostingClassifier(
    n_estimators=200,     
    learning_rate=0.05,
    max_depth=3,
    subsample=0.8,        # stochastic boosting speeds & regularizes
    random_state=42
)
gb_baseline.fit(X_train, y_train)
results_gb = evaluate_model(gb_baseline, X_train, y_train, X_test, y_test, "Gradient Boosting (baseline)")




In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import GradientBoostingClassifier
from scipy.stats import randint, uniform

gb_best = GradientBoostingClassifier(random_state=42)
gb_dist = {
    "n_estimators": randint(150, 450),
    "learning_rate": uniform(0.02, 0.15),  # ~0.02–0.17
    "max_depth": randint(2, 6),
    "subsample": uniform(0.7, 0.3),        # 0.7–1.0
    "min_samples_leaf": randint(1, 10)
}

gb_search = RandomizedSearchCV(
    gb, gb_dist, n_iter=30, scoring="roc_auc",
    cv=5, n_jobs=-1, random_state=42, verbose=0
)
gb_search.fit(X_train, y_train)
gb_best = gb_search.best_estimator_
print("GB best params:", gb_search.best_params_, "CV AUC:", round(gb_search.best_score_, 3))
results_gb_best = evaluate_model(gb_best, X_train, y_train, X_test, y_test, "Gradient Boosting (tuned)")


In [ ]:
print(pd.concat(
    [results_gb, results_gb_best],
    axis=0,
    ignore_index=True
).set_index("Model"))

**Comment:** The fine tuned Gradient Boosting model performs better across all the metrics. 

In [ ]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier

ada_baseline = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1, random_state=42),  # stump
    n_estimators=300,
    learning_rate=0.1,
    random_state=42
)
ada_baseline.fit(X_train, y_train)
results_ada = evaluate_model(ada_baseline, X_train, y_train, X_test, y_test, "AdaBoost (baseline)")


In [ ]:
ada = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(random_state=42),
    random_state=42
)

ada_dist = {
    "estimator__max_depth": randint(1, 3),
    "n_estimators": randint(200, 600),
    "learning_rate": uniform(0.02, 0.3)
}

ada_search = RandomizedSearchCV(
    ada,
    ada_dist,
    n_iter=25,
    scoring="roc_auc",
    cv=5,
    n_jobs=-1,
    random_state=42
)
ada_search.fit(X_train, y_train)

ada_best = ada_search.best_estimator_
print("AdaBoost best params:", ada_search.best_params_)
print("Mean CV ROC-AUC:", round(ada_search.best_score_, 3))

results_ada_best = evaluate_model(ada_best, X_train, y_train, X_test, y_test, "AdaBoost (tuned)")


In [ ]:
print(pd.concat(
    [results_ada, results_ada_best],
    axis=0,
    ignore_index=True
).set_index("Model"))

**Comment:** The fine tuned model performs better across all the metrics.

In [ ]:
pip install xgboost

In [ ]:
from xgboost import XGBClassifier

xgb_clf = XGBClassifier(
    use_label_encoder=False,
    eval_metric="auc",
    random_state=42,
    n_estimators=400,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,      # feature sampling
    scale_pos_weight=y_train.value_counts()[0] / y_train.value_counts()[1],  # handle 80/20 imbalance
    tree_method="hist",        # fast histogram-based algorithm
    n_jobs=-1    
)

models = {"XGBoost":xgb_clf}


# Train & evaluate
results = []
for name, model in models.items():
    # Fit
    model.fit(X_train, y_train)

    # Predictions
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]  # probability of class=1 (conversion)

results_xg = evaluate_model(xgb_clf, X_train, y_train, X_test, y_test, "XGBoost (baseline)")

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

xgb_dist = {
    "n_estimators": randint(200, 600),
    "learning_rate": uniform(0.02, 0.1),
    "max_depth": randint(2, 6),
    "subsample": uniform(0.7, 0.3),
    "colsample_bytree": uniform(0.6, 0.4)
}

search = RandomizedSearchCV(
    xgb_clf, xgb_dist, n_iter=25, scoring="roc_auc",
    cv=5, n_jobs=-1, random_state=42
)
search.fit(X_train, y_train)
best_xgb = search.best_estimator_
results_xg_best = evaluate_model(best_xgb, X_train, y_train, X_test, y_test, "XGBoost (tuned)")

# Feature importance by 'gain' 
booster = best_xgb.get_booster()
importance = booster.get_score(importance_type="gain")

# Convert to a tidy DataFrame
importance_df = (
    pd.DataFrame(list(importance.items()), columns=["Feature", "Gain"])
    .sort_values(by="Gain", ascending=False)
)

print("\n=== Top 15 features by Gain (XGBoost tuned) ===")
print(importance_df.head(15).to_string(index=False))



In [ ]:
print(pd.concat(
    [results_xg, results_xg_best],
    axis=0,
    ignore_index=True
).set_index("Model"))

**Comment:** The fine tuned model performs better across all the metrics but Recall.

In [ ]:
# Standardize names first 
results_log = (
    results_df.loc[["LogReg (no penalty)"]]
              .reset_index()
              .rename(columns={"index": "Model", "ROC-AUC": "ROC_AUC", "CV_ROC-AUC": "CV_ROC_AUC"})
)

# Keep only the common metric columns
common_cols = ["Model", "Accuracy", "Precision", "Recall", "F1", "ROC_AUC", "CV_ROC_AUC"]
results_log = results_log.reindex(columns=common_cols)


In [ ]:
print(results_log)

In [ ]:
print("\n=== Model Comparison Summary ===")

print(pd.concat(
    [results_dt_best, results_rf_best, results_gb_best, results_ada_best, results_xg_best],
    axis=0,
    ignore_index=True
).set_index("Model"))

**Comparison of various techniques and their relative performance based on chosen Metric (Measure of success):** 

**Overall predictive power:**

* The ensemble models (XGBoost, Gradient Boosting, Random Forest, AdaBoost) all reach very high AUC scores (≈ 0.92–0.96), indicating excellent discrimination between default and non-default clients.

* Random Forest performs very well but slightly below boosting — consistent with theory (boosting reduces bias more effectively).

* Logistic Regression and the Decision Tree are strong baselines but clearly less powerful in non-linear decision boundaries.

Conclusion: Boosting and bagging ensembles clearly capture complex, nonlinear interactions that linear models miss.

**Accuracy and recall balance**

* XGBoost and Gradient Boosting achieve the best balance of precision and recall.

* XGBoost has slightly higher recall (0.76) but lower precision than Gradient Boosting.

* Random Forest performs almost as well (AUC = 0.95) but sacrifices recall for precision (it misses more defaulters).

* AdaBoost still performs decently (AUC ≈ 0.92) but quite low precision and recall.

Conclusion: Gradient-based ensembles (especially XGBoost) offer the strongest recall–precision trade-off.

**Interpretability and baseline**

* The Decision Tree is much simpler and interpretable; its drop in AUC is expected but it’s useful for explaining decision rules to non-technical audiences.

* The Logistic Regression failed mainly because the data are non-linear and include complex feature interactions (e.g., debt-to-income, delinquencies, credit age). Its higher recall (0.80) but low precision (0.58) means it flags almost everyone as defaulter—very conservative but not practically useful.

Conclusion: Logistic regression remains a benchmark for interpretability, but for this dataset it underfits.

**Proposal for the final solution design:** 


* Ensemble methods clearly improve default detection accuracy without excessive false positives.

* XGBoost’s high recall means fewer missed defaulters → lower financial risk.

* Gradient Boosting’s higher precision means fewer good clients wrongly rejected → better customer experience.

* Combining interpretability tools (feature importance, SHAP) with XGBoost gives both performance and explainability, meeting model-risk governance standards.

**Recommendation:** Model XGBoost with further steps to improve explanability.

**EXTRA STEPS:**

* We first try to evaluate more precisely the trade-off default detection - intepretability by measuring the expected losses for missing defaults (false negative). 

ASSUMPTION: We start by assuming that the bank can recover 40% of defaulted loan value. We can adjust this later on. 

In [ ]:


# Parameters 
LGD = 0.40                
EAD_COL = "LOAN"           

# Helper 
def missed_default_value(y_true, y_pred, ead, lgd=0.40):
    """Return total missed-default £loss and FN count."""
    mask_fn = (y_true == 1) & (y_pred == 0)
    total_loss = float((ead[mask_fn] * lgd).sum())
    count_fn = int(mask_fn.sum())
    avg_ead = float(ead[mask_fn].mean()) if count_fn > 0 else 0.0
    return total_loss, count_fn, avg_ead

# Collect models here (use the tuned ones you trained) 
models = []
if "best_dt"  in globals(): models.append( ("Decision Tree (tuned)",  best_dt) )
if "best_rf"  in globals(): models.append( ("Random Forest (tuned)",  best_rf) )
if "gb_best"  in globals(): models.append( ("Gradient Boosting (tuned)", gb_best) )
if "ada_best" in globals(): models.append( ("AdaBoost (tuned)",       ada_best) )
if "best_xgb" in globals(): models.append( ("XGBoost (tuned)",        best_xgb) )

# pick logistic model variable name (pipe or best_l1 or log_reg)
if "pipe_plain" in globals():
    models.append(("Logistic Regression (no penalty)", pipe_plain))

# Compute loss per model on the test set 
rows = []
ead_test = X_test[EAD_COL]  

for name, model in models:
    y_pred = model.predict(X_test)
    total_loss, fn_count, avg_ead = missed_default_value(y_test.values, y_pred, ead_test.values, lgd=LGD)
    rows.append({
        "Model": name,
        "FN (missed defaulters)": fn_count,
        f"Total Missed Loss (£) @ LGD={LGD:.0%}": total_loss,
        "Avg EAD of FN (£)": avg_ead
    })

missed_loss_df = pd.DataFrame(rows).sort_values(by=f"Total Missed Loss (£) @ LGD={LGD:.0%}", ascending=True)
pd.set_option("display.float_format", lambda x: f"{x:,.0f}")
print(missed_loss_df)


In [ ]:
# Evaluate the expected losses for false negative at different thresholds for XGBoost model

def missed_loss_at_threshold(model, X, y_true, ead, threshold=0.5, lgd=0.40):
    if hasattr(model, "predict_proba"):
        p = model.predict_proba(X)[:, 1]
    else:
        s = model.decision_function(X)
        p = 1 / (1 + np.exp(-s))
    y_pred = (p >= threshold).astype(int)
    return missed_default_value(y_true, y_pred, ead, lgd=lgd)

thresholds = [0.50, 0.45, 0.40, 0.35]
for thr in thresholds:
    loss, fn, _ = missed_loss_at_threshold(best_xgb, X_test, y_test.values, ead_test.values, threshold=thr, lgd=LGD)
    print(f"XGBoost @ threshold={thr:.2f} → FN={fn}, Missed Loss £={loss:,.0f}")


**Comment:** We can us stronger threshold in the XGBoost model and outperform the Logistic model in this dimension.

We now estimate the opportunity cost of the missed "good" loans.

ASSUMPTION: We consider an annualized amount and an annual interest rate of 4% on the loan. This can be adjusted as a parameter.

In [ ]:

# Assumptions 
INTEREST_RATE = 0.04   # 4% of loan amount (annual)
EAD_COL = "LOAN"       

# Helper: Missed opportunity (False Positives) valued as interest on LOAN 
def missed_opportunity_interest(y_true, y_pred, ead_series, interest_rate=0.04):

    y_true = pd.Series(y_true)
    y_pred = pd.Series(y_pred, index=y_true.index)
    ead = pd.Series(ead_series, index=y_true.index)

    mask_fp = (y_true == 0) & (y_pred == 1)
    per_fp_value = ead[mask_fp] * float(interest_rate)
    total_loss = float(per_fp_value.sum())
    count_fp   = int(mask_fp.sum())
    avg_ead    = float(ead[mask_fp].mean()) if count_fp > 0 else 0.0
    return total_loss, count_fp, avg_ead

# Collect your fitted models (edit names/vars if needed)
models = []
if "best_dt"  in globals(): models.append(("Decision Tree (tuned)",  best_dt))
if "best_rf"  in globals(): models.append(("Random Forest (tuned)",  best_rf))
if "gb_best"  in globals(): models.append(("Gradient Boosting (tuned)", gb_best))
if "ada_best" in globals(): models.append(("AdaBoost (tuned)",       ada_best))
if "best_xgb" in globals(): models.append(("XGBoost (tuned)",        best_xgb))

if "pipe_plain" in globals():
    models.append(("Logistic Regression (no penalty)", pipe_plain))

# EAD vector aligned to y_test 
if EAD_COL in X_test.columns:
    ead_test = X_test[EAD_COL]
else:
    # fallback to a constant exposure if LOAN not present in X_test
    ead_test = pd.Series(np.full(len(y_test), 50_000.0), index=y_test.index)

# Build Missed Opportunity table 
rows = []
for name, model in models:
    y_pred = model.predict(X_test)
    total_loss, fp_count, avg_ead = missed_opportunity_interest(
        y_test, y_pred, ead_test, interest_rate=INTEREST_RATE
    )
    rows.append({
        "Model": name,
        "FP (missed opportunities)": fp_count,
        f"Total Missed Opportunity (£) @ {INTEREST_RATE:.0%} of LOAN": total_loss,
        "Avg EAD of FP (£)": avg_ead
    })

missed_opp_df = pd.DataFrame(rows).sort_values(
    by=f"Total Missed Opportunity (£) @ {INTEREST_RATE:.0%} of LOAN"
)

pd.set_option("display.float_format", lambda x: f"{x:,.0f}")
print("\n=== Missed Opportunity (False Positives) — Valued as Interest on LOAN ===")
print(missed_opp_df)


In [ ]:
# Merge and sum monetary columns
total_cost_df = pd.merge(missed_loss_df, missed_opp_df, on="Model", how="outer")

money_cols = [c for c in total_cost_df.columns if c.startswith("Total Missed")]
total_cost_df["Total Estimated Cost (£)"] = total_cost_df[money_cols].sum(axis=1)

print("\n=== Combined Cost: Missed Defaults + Missed Opportunities ===")
print(total_cost_df.sort_values("Total Estimated Cost (£)").reset_index(drop=True))


**Summary:**

We computed:

- Loss from missed defaults = FN × EAD × LGD

- Loss from missed opportunities = FP × EAD × 4 %

- Sum them → Total Estimated Cost.

We conclude that

* Even though Logistic Regression catches more defaulters (lower FN), it over-rejects profitable borrowers. 

* XGBoost, with slightly higher FN but much lower FP, ends up having the lowest total expected loss (£735k).

* The asymmetry of costs (40 % LGD vs 4 % interest) means false negatives are roughly 10 times more expensive than false positives. Hence, models that minimize false negatives (like Logistic and XGBoost) dominate, even if they over-reject a few good borrowers.

**Robustness Check:** We check how our conclusion and ranking of models based on expected net losses is robust to our two parameters choice.

In [ ]:
LGD_vals = np.linspace(0.2, 0.8, 7)        # 20%–80% loss given default
r_vals   = np.linspace(0.02, 0.08, 7)      # 2%–8% interest rate

records = []
for lgd in LGD_vals:
    for r in r_vals:
        for i, row in missed_loss_df.iterrows():  # FN results table
            name = row["Model"]
            fn_loss = row[f"Total Missed Loss (£) @ LGD=40%"] / 0.40 * lgd  # scale proportionally
            fp_loss = missed_opp_df.loc[missed_opp_df["Model"] == name,
                                        f"Total Missed Opportunity (£) @ 4% of LOAN"].values[0] / 0.04 * r
            total = fn_loss + fp_loss
            records.append({"Model": name, "LGD": lgd, "InterestRate": r, "TotalCost": total})

sensitivity = pd.DataFrame(records)

In [ ]:
winners = (sensitivity
           .groupby(["LGD","InterestRate"])
           .apply(lambda d: d.loc[d["TotalCost"].idxmin(), ["Model","TotalCost"]])
           .reset_index())

In [ ]:
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap

# winners: DataFrame with columns ["LGD","InterestRate","Model"]
# Build the pivot of model names
pivot = winners.pivot(index="LGD", columns="InterestRate", values="Model")
pivot = pivot.sort_index().sort_index(axis=1) 

# 1) Factorize once over the entire matrix (consistent codes)
codes, labels = pd.factorize(pivot.values.ravel(), sort=True)  # labels = unique model names (Index)
matrix = codes.reshape(pivot.shape)

# 2) Discrete colormap with exactly len(labels) colors
N = len(labels)
cmap = plt.get_cmap("tab20", N)    


fig, ax = plt.subplots(figsize=(9, 5.5))
im = ax.imshow(matrix, cmap=cmap, vmin=0, vmax=N-1, aspect="auto")

# Axis ticks/labels
ax.set_xticks(range(pivot.shape[1]))
ax.set_xticklabels([f"{c:.0%}" for c in pivot.columns])
ax.set_yticks(range(pivot.shape[0]))
ax.set_yticklabels([f"{r:.0%}" for r in pivot.index])
ax.set_xlabel("Interest Rate")
ax.set_ylabel("Loss Given Default")
ax.set_title("Model with Minimum Expected Total Cost")

# 3) Legend built from the same cmap indices used in the image
patches = [mpatches.Patch(color=cmap(i), label=str(labels[i])) for i in range(N)]
leg = ax.legend(handles=patches, title="Model", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=True)

plt.tight_layout()
plt.show()



**Conclusion:** The matrix provide a potentially useful map to identify the most cost effective model. It is evident that when the profit per performing loan increases, it becomes more valuable to avoid false positives,hence XGBoost (with higher precision) becomes the preferred model. Conversely, when losses from defaults increase (high LGD), avoiding false negatives becomes more important, so Logistic Regression (with higher recall) minimizes total cost.

**EXTRA STEP:**

If we decide to use XGBoost, it might be important to enhance its interpretability. We now provide extra informations in this direction.

In [ ]:
# Gain-based feature importance (most informative globally)
gain_importance = best_xgb.get_booster().get_score(importance_type="gain")
fi = pd.Series(gain_importance).sort_values(ascending=True).tail(15)

plt.figure(figsize=(8,6))
fi.plot(kind="barh", color="royalblue")
plt.title("Top 15 Features by Gain Importance (XGBoost)")
plt.xlabel("Average Gain")
plt.tight_layout()
plt.show()


**Comment:** The plot helps us to understand how much predictive value each of the top 15 features in XGBoost model adds.

* Missing data indicators (*_missing) play a disproportionately large role — the model uses them to detect risky or opaque applicants.

* Debt burden and delinquency variables remain the core behavioural risk factors.

* Job-type and credit-history metrics add marginal refinements but much less incremental predictive gain.

In [ ]:
from sklearn.inspection import PartialDependenceDisplay

features_to_plot = ["DEBTINC_missing","DEBTINC","VALUE_missing","DELINQ","DEROG"]

# Capture the display (and figure it creates)
disp = PartialDependenceDisplay.from_estimator(
    best_xgb, X_test, features_to_plot,
    kind="average", grid_resolution=50, random_state=42
)


fig = disp.figure_
fig.suptitle("Partial Dependence — effect on default probability", fontsize=12)
fig.subplots_adjust(hspace=0.4, top=0.9)  # adjust vertical spacing & leave room for title
plt.show()


The figure above shows how the predicted probability of default changes as a given variable varies, holding all other features constant (on average). For instance:

* When debt-to-income data is missing, the model predicts a higher default probability, even when controlling for everything else. This confirms what the feature importance plot showed — missingness itself is a strong red flag.
* Conditional on other features, the raw debt-to-income ratio adds only limited incremental predictive power compared to the “missing” indicator. Possible reason is that the effect may be nonlinear or already captured by other correlated variables (e.g. DEROG, DELINQ).
* Missing property value is highly predictive of risk — applicants without collateral or appraisal info are far more likely to default.
* Past delinquency behaviour is one of the strongest direct indicators of future default.
* Default probability climbs quickly with the number of derogatory records, plateauing around 4–5. In other words, more negative credit events → higher risk, with diminishing returns after a threshold.

In [ ]:
pip install shap


In [ ]:
import shap
explainer = shap.Explainer(best_xgb)
shap_values = explainer(X_test)

# Global summary plot
shap.summary_plot(shap_values, X_test, plot_type="bar", max_display=15)


In [ ]:
shap.summary_plot(shap_values, X_test, max_display=15)


**SHAP (SHapley Additive exPlanations) analysis:** This can provide more information on which features drive predictions most strongly and in what direction they influence the predicted default probability. General interpretation of the plots above is that XGBoost’s decisions rely heavily on debt burden, information completeness, and credit history — a pattern consistent with what your feature-importance and PDP analyses already showed.

More precisely:

* Debt burden dominates: Borrowers with high debt-to-income ratios — or missing debt info — contribute the most to the model’s risk predictions.

* Credit behaviour matters: Delinquencies, derogatory marks, and short credit histories sharply increase predicted default probability.

* Information quality and stability signals help: Missing values, short job tenure, and frequent credit inquiries all tilt predictions toward “high risk.”

* Collateral & employment moderate risk: High property values, longer tenure, and well-documented loans reduce the model’s default estimate.


**IMPORTANT Observation:**

- For certain binary or categorical variables (like DEBTINC_missing, DEROG, or VALUE_missing), the SHAP values split neatly into two clusters — some positive (pushing risk up), some negative (pushing risk down). This reflects either existing feature interactions — i.e., the effect of that variable depends on other features’ values or in general a strong segmentation power of that feature. For instance, DEBTINC_missing seems to split the observations into two distinct clusters

**Example of use OF SHAP for an individual case:** We can build a shap analysis on individual observations to (i) improve explanability of the model and (ii) to test regulations. See example below.

In [ ]:
# Choose a single observation
idx = 10   
shap.waterfall_plot(shap_values[idx], max_display=15)


**Comment on example:**

| Step | Feature             | SHAP (Δ log-odds) | Cumulative log-odds | Probability Default| Interpretation |
|------|---------------------|-------------------|----------------------|-------------------|----------------|
| Base | –                   | –                 | 0.694                | 0.667             | Average applicant’s default probability ≈ 66.7% |
| 1    | VALUE = 94,726      | −1.12             | −0.426               | 0.395             | High property value lowers risk substantially (around 26% drop)|
| 2    | CLNO = 31           | −0.97             | −1.396               | 0.198             | Extensive credit history reduces default risk |
| 3    | DEBTINC_missing = 0 | −0.95             | −2.346               | 0.087             | Complete DTI data lowers uncertainty and risk |
| 4    | DELINQ = 0          | −0.90             | −3.246               | 0.037             | No delinquencies strongly reduce default risk |
| 5    | NINQ = 1            | −0.69             | −3.936               | 0.019             | Few recent inquiries → lower credit pressure |
| 6    | YOJ = 1             | −0.41             | −4.346               | 0.013             | Employment stability lowers default likelihood |
| 7    | DEROG = 0           | −0.31             | −4.656               | 0.009             | No derogatory marks further reduce risk |
| 8    | MORTDUE = 71,264    | −0.31             | −4.966               | 0.007             | Moderate mortgage balance = manageable leverage |
| 9    | JOB_Other = True    | +0.28             | −4.686               | 0.009             | Job category “Other” adds slight risk |
| 10   | JOB_Office = False  | +0.20             | −4.486               | 0.011             | Not in “Office” job increases risk marginally |
| 11   | CLAGE = 177.26      | −0.19             | −4.676               | 0.009             | Older credit history lowers default probability |
| 12   | REASON_HomeImp = F  | −0.17             | −4.846               | 0.008             | Not a home improvement loan slightly safer |
| 13   | VALUE_missing = 0   | −0.09             | −4.936               | 0.007             | No missing property info → reduced uncertainty |
| 14   | JOB_Sales = False   | −0.08             | −5.016               | 0.007             | Not in “Sales” job (lower volatility) |
| ...  | Other small effects | −0.08             | −5.097               | **0.006**         | Final predicted default probability ≈ **0.6%** |



**Comment:**

The SHAP analysis of the tuned XGBoost model highlights Debt-to-Income ratio (DEBTINC) as the most influential predictor of loan default, followed closely by the indicator for missing DEBTINC values. This suggests that both the level of indebtedness and the absence of reliable income data are strong signals of default risk. Other key drivers include number of delinquencies (DELINQ) and credit age (CLAGE), both of which reflect an applicant’s credit history and repayment behaviour. Variables such as loan amount (LOAN), number of credit lines (CLNO), and mortgage due (MORTDUE) also contribute meaningfully, indicating that higher overall exposure increases the likelihood of default.

In contrast, employment-related features (e.g., YOJ, JOB_Office, JOB_Other) and loan purpose (REASON_HomeImp) have a relatively lower impact on predictions, implying that credit and balance sheet indicators dominate the model’s decision-making process. Overall, the SHAP results confirm that the XGBoost model primarily relies on financial leverage, credit behaviour, and repayment history to assess default risk—an economically intuitive and interpretable outcome that aligns with standard credit risk theory.

### **RERUN of models without JOB for possible regulatory infringement**

We now run the models after excluding JOB from the dataset. As JOB could be a problematic feature for a regulator so, as robustness check, we run all the models again excluding this feature and pick a model under this constraint. 

In [ ]:
df_nojob = df.drop(columns=["JOB_Office", "JOB_Other", "JOB_ProfExe", "JOB_Sales", "JOB_Self"])
df_nojob.head()

For compactness, we rerun all the models together, using one single code snippet.

In [ ]:
TARGET = "BAD"
Xn = df_nojob.drop(columns=[TARGET]).copy()
yn = df_nojob[TARGET].copy()

from sklearn.model_selection import train_test_split
Xn_train, Xn_test, yn_train, yn_test = train_test_split(
    Xn, yn, test_size=0.30, random_state=42, stratify=yn
)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

num_cols = Xn.select_dtypes(include=["number"]).columns
cat_cols = Xn.select_dtypes(include=["object", "category", "bool"]).columns

preprocessor_n = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(with_mean=False), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=True), cat_cols),
    ],
    remainder="drop"  
)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from xgboost import XGBClassifier

models = {
    "Logistic (L1)": LogisticRegression(penalty="l1", solver="liblinear", random_state=42, max_iter=2000, class_weight="balanced"),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42, n_jobs=-1, class_weight="balanced_subsample"),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "AdaBoost": AdaBoostClassifier(random_state=42),
    "XGBoost": XGBClassifier(random_state=42, eval_metric="logloss", tree_method="hist", n_jobs=-1),
}

param_grids = {
    "Decision Tree": {"model__max_depth": [3,5,7]},
    "Random Forest": {"model__n_estimators": [150,250], "model__max_depth": [8,12]},
    "Gradient Boosting": {"model__n_estimators": [100,200], "model__learning_rate": [0.05,0.1]},
    "AdaBoost": {"model__n_estimators": [100,200], "model__learning_rate": [0.5,1.0]},
    "XGBoost": {"model__n_estimators": [200,300], "model__max_depth": [3,5], "model__learning_rate": [0.05,0.1]},
}


In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
import pandas as pd

pd.reset_option('display.float_format')        # back to default

pd.options.display.float_format = None

results = []
best_models_n = {}

for name, model in models.items():
    pipe = Pipeline([("preprocessor", preprocessor_n), ("model", model)])
    grid = param_grids.get(name)

    if grid:
        search = GridSearchCV(pipe, grid, cv=5, scoring="roc_auc", n_jobs=-1, refit=True)
        search.fit(Xn_train, yn_train)
        best = search.best_estimator_
    else:
        best = pipe.fit(Xn_train, yn_train)

    # sanity prints per model
    match_rate = (yn_test.values == best.predict(Xn_test)).mean()
    auc = roc_auc_score(yn_test, getattr(best, "predict_proba")(Xn_test)[:,1])
    print(f"[{name}] match_rate={match_rate:.3f}  AUC={auc:.3f}")
    print(confusion_matrix(yn_test, best.predict(Xn_test)))

    y_pred = best.predict(Xn_test)
    y_prob = best.predict_proba(Xn_test)[:,1]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(yn_test, y_pred),
        "Precision": precision_score(yn_test, y_pred, zero_division=0),
        "Recall": recall_score(yn_test, y_pred, zero_division=0),
        "F1": f1_score(yn_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(yn_test, y_prob)
    })
    best_models_n[name] = best

results_df_nojob= pd.DataFrame(results).sort_values("ROC-AUC", ascending=False).round(3)
print("\n=== Results on df_nojob ===")
print(results_df.round(3))


In [ ]:
# FULL dataset: build X/y, split, fresh preprocessor
TARGET = "BAD"

Xf = df.drop(columns=[TARGET]).copy()
yf = df[TARGET].copy()

from sklearn.model_selection import train_test_split, GridSearchCV
Xf_train, Xf_test, yf_train, yf_test = train_test_split(
    Xf, yf, test_size=0.30, random_state=42, stratify=yf
)

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import pandas as pd

num_f = Xf.select_dtypes(include=["number"]).columns
cat_f = Xf.select_dtypes(include=["object","category","bool"]).columns

preprocessor_f = ColumnTransformer(
    [("num", StandardScaler(with_mean=False), num_f),
     ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=True), cat_f)],
    remainder="drop"
)

#  Reuse your same `models` and `param_grids` dicts 
results_full = []
best_models_full = {}

for name, model in models.items():
    pipe = Pipeline([("preprocessor", preprocessor_f), ("model", model)])
    grid = param_grids.get(name)

    if grid:
        search = GridSearchCV(pipe, grid, cv=5, scoring="roc_auc", n_jobs=-1, refit=True)
        search.fit(Xf_train, yf_train)
        best = search.best_estimator_
    else:
        best = pipe.fit(Xf_train, yf_train)

    y_pred = best.predict(Xf_test)
    y_prob = best.predict_proba(Xf_test)[:, 1]

    results_full.append({
        "Model": name,
        "Accuracy": accuracy_score(yf_test, y_pred),
        "Precision": precision_score(yf_test, y_pred, zero_division=0),
        "Recall": recall_score(yf_test, y_pred, zero_division=0),
        "F1": f1_score(yf_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(yf_test, y_prob),
    })
    best_models_full[name] = best

results_df_full = pd.DataFrame(results_full).sort_values("ROC-AUC", ascending=False).round(3)
print("=== Results on df (full) ===")
print(results_df_full)


In [ ]:
# Ensure both exist: results_df_full (from code above) and results_df_nojob (from earlier)
comparison = results_df_nojob.merge(
    results_df_full, on="Model", suffixes=("_noJOB", "_full")
)

# Compute deltas
for m in ["Accuracy","Precision","Recall","F1","ROC-AUC"]:
    comparison[f"Δ_{m}"] = comparison[f"{m}_noJOB"] - comparison[f"{m}_full"]

# Sort by AUC change and display
comparison = comparison.sort_values("Δ_ROC-AUC", ascending=False).round(3)
print("\n=== Comparison (noJOB - full) ===")
print(comparison[[
    "Model",
    "ROC-AUC_noJOB","ROC-AUC_full","Δ_ROC-AUC",
    "F1_noJOB","F1_full","Δ_F1",
    "Accuracy_noJOB","Accuracy_full","Δ_Accuracy"
]])


### **FINAL CONCLUSION**: 

* The models perform virtually identically with or without JOB feature. The loss across the metrics are sufficiently small to not affect the decision substantially and more importantly the "costs". I would recommend to use XGBoost model, excluding feature JOB, and using SHAP to interpret and justify the algorithm conclusions. 

In [ ]:
xgb_results = results_df_nojob[results_df_nojob["Model"].str.contains("XGBoost", case=False)]
print(xgb_results)


In [ ]:

xgb_key = next(k for k in best_models_n.keys() if "xgboost" in k.lower())
best_xgb = best_models_n[xgb_key]   
print(f"Retrieved model: {xgb_key}")
best_xgb


In [ ]:

xgb_clf = best_xgb.named_steps["model"]

# (a) All sklearn-style params of the XGBClassifier:
xgb_params = xgb_clf.get_params()
xgb_params  # big dict

# (b) The core Booster params (what XGBoost itself uses):
booster_params = xgb_clf.get_xgb_params()


In [ ]:
import json

keys_of_interest = [
    "n_estimators", "learning_rate", "max_depth",
    "subsample", "colsample_bytree", "min_child_weight",
    "gamma", "reg_alpha", "reg_lambda", "random_state"
]

pretty = {k: xgb_params.get(k, None) for k in keys_of_interest}
print(json.dumps(pretty, indent=2))


In [ ]:

xgb = best_xgb.named_steps["model"]

print(xgb.get_xgb_params())    # booster params actually used


In [ ]:
# Extract feature names (excluding target)
features = df_nojob.drop(columns=["BAD"]).columns.tolist()

# Format as a Markdown bullet list
md_list = "\n".join([f"- `{f}`" for f in features])
print(md_list)


# **FINAL REPORT** #

## **1. Executive Summary** 

This project developed a machine-learning model to predict loan default risk for a retail bank’s home-equity portfolio. The dataset (HMEQ) included 5,960 records with 12 predictors on applicants’ financials, employment, credit history, and collateral. 

After extensive exploratory data analysis and model benchmarking, the final selected model is an **XGBoost (tuned) classifier** trained on preprocessed data excluding the JOB variable (to mitigate bias and ensure interpretability compliance as discussed later). Here a summary:

Model: **XGBoost (tuned) classifier** 

Key model parameters:

| Parameter             | Value     | Purpose                                        |
|-----------------------|-----------|------------------------------------------------|
| `n_estimators`        | 300       | Number of boosting rounds                      |
| `learning_rate`       | 0.10      | Shrinkage factor controlling learning speed    |
| `max_depth`           | 5         | Maximum depth of individual trees              |
| `subsample`           | 1         | Fraction of data used per boosting round       |
| `colsample_bytree`    | 1         | Fraction of features used per tree             |
| `min_child_weight`    | 1         | Minimum sum of instance weight in a child node |
| `gamma`               | 0         | Minimum loss reduction required for a split    |
| `reg_alpha`           | 0         | L1 regularization term on weights              |
| `reg_lambda`          | 1         | L2 regularization term on weights              |
| `random_state`        | 42        | Reproducibility seed                           |


Model Features: 

| Feature          | Description                        |
|------------------|------------------------------------|
| LOAN             | Loan amount requested              |
| MORTDUE          | Current mortgage due               |
| VALUE            | Property value                     |
| YOJ              | Years at current job               |
| DEROG            | Number of major derogatory reports |
| DELINQ           | Number of delinquent credit lines  |
| CLAGE            | Age of oldest credit line (months) |
| NINQ             | Number of recent credit inquiries  |
| CLNO             | Number of existing credit lines    |
| DEBTINC          | Debt-to-income ratio               |
| REASON_HomeImp   | Loan reason: home improvement      |
| VALUE_missing    | Missing flag for VALUE             |
| MORTDUE_missing  | Missing flag for MORTDUE           |
| DEBTINC_missing  | Missing flag for DEBTINC           |

Key model performance on the test set:

| Model   | Accuracy | Precision | Recall | F1    | ROC-AUC |
|---------|----------|-----------|--------|------ |---------|
| XGBoost | 0.912    | 0.844     | 0.683  | 0.755 | 0.955   |

Feature-importance and SHAP analyses reveal that:
- Debt-to-Income ratio (DEBTINC) and missing DEBTINC data are the strongest default indicators.
- Credit behaviour — delinquencies (DELINQ), derogatory reports (DEROG), and short credit history (CLAGE) — sharply increase default probability.
- Collateral strength (VALUE, MORTDUE) and employment stability (YOJ) reduce default risk.
- Missing information itself (e.g., missing property value) signals higher risk.

The model achieves high discrimination (ROC-AUC ≈ 0.96) while maintaining interpretability via SHAP explanations suitable for regulatory justification under the Equal Credit Opportunity Act (ECOA).


## Model Selection Strategy


To identify the best predictive model, several algorithms were trained and evaluated on the dataset **excluding the `JOB` variable (`df_nojob`)**:  
**Logistic Regression (L1-regularized), Decision Tree, Random Forest, Gradient Boosting, AdaBoost,** and **XGBoost**.

Performance was assessed using five standard classification metrics: **Accuracy**, **Precision**, **Recall**, **F1-score**, and **ROC-AUC**.

- **Accuracy** → overall correct classifications.  
- **Precision** → proportion of predicted positives that are actual positives (avoiding false alarms).  
- **Recall** → proportion of actual positives correctly identified (avoiding missed defaults).  
- **F1-score** → harmonic mean of Precision and Recall (balance measure).  
- **ROC-AUC** → evaluates the model’s ranking ability across thresholds.

---

### Model Comparison

| Model             | Accuracy | Precision | Recall    | F1        | ROC-AUC  |
|-------------------|----------|-----------|-----------|-----------|----------|
| Logistic (L1)     | 0.848    | 0.589     | 0.796     | 0.677     | 0.890    |
| Decision Tree     | 0.875    | 0.769     | 0.532     | 0.629     | 0.838    |
| Random Forest     | 0.893    | 0.750     | 0.697     | 0.723     | 0.946    |
| Gradient Boosting | 0.903    | 0.812     | 0.667     | 0.732     | 0.941    |
| AdaBoost          | 0.881    | 0.814     | 0.527     | 0.639     | 0.908    |
| **XGBoost**       | **0.912**| **0.844** | **0.683** | **0.755** | **0.955**|

---

### Model Choice and Rationale

The **XGBoost model** achieved the **highest overall performance**, with the best **Accuracy (0.912)** and **ROC-AUC (0.955)**, as well as a strong balance between **Precision (0.844)** and **Recall (0.683)**.  
This balance indicates that the model effectively identifies high-risk applicants (true positives) while limiting false alarms (false positives).  
Given the credit risk context—where both missed defaulters and false rejections are costly—this trade-off is especially valuable.

Additionally, XGBoost’s gradient boosting structure and built-in regularization help prevent overfitting and capture **nonlinear interactions** among financial variables—something simpler linear models cannot easily achieve.  
The model’s **high ROC-AUC** also provides significant **headroom for calibration**: by tuning the decision threshold or adjusting class weights, recall can be increased further when the cost of missed defaults outweighs that of false positives.  
This flexibility makes XGBoost particularly well-suited for real-world deployment, where the optimal precision–recall trade-off may vary with changing market or policy conditions.


---

### Why Other Models Were Not Chosen

Although several models performed reasonably well, none achieved the same combination of **accuracy, robustness, and interpretability** as XGBoost:

- **Logistic Regression (L1-regularized):**  
  The logistic model achieved a relatively high **Recall (0.796)** but low **Precision (0.589)** and **ROC-AUC (0.890)**.  
  This means it captured many true defaulters but at the cost of numerous false positives—potentially rejecting creditworthy applicants.  
  Its linear structure also limits its ability to represent complex, nonlinear relationships among variables such as `DEBTINC`, `VALUE`, or `DELINQ`.  
  Logistic regression remains highly interpretable and suitable when model transparency is the overriding priority, but in this case its **predictive balance and discrimination power** were weaker than the ensemble methods.

- **Random Forest:**  
  The Random Forest model ranked second overall (**ROC-AUC = 0.946**) and offered solid recall, but it lagged slightly in **Precision** and **Accuracy** compared to XGBoost.  
  While Random Forests reduce variance effectively through averaging, they can be **less sensitive to small, systematic patterns** in the data and **less efficient** in handling class imbalance.  
  In contrast, XGBoost’s sequential boosting framework allows each new tree to focus specifically on prior misclassifications, improving boundary precision and bias–variance trade-offs.  
  XGBoost also converges faster and integrates better with interpretability tools like **SHAP**[^4], providing clearer insight into individual predictions.

- **Other Ensemble Models (Gradient Boosting, AdaBoost):**  
  Both Gradient Boosting and AdaBoost achieved strong results but did not match XGBoost’s combination of **regularization**, **training efficiency**, and **AUC performance**.  
  Their slightly lower recall and higher error variance made them less reliable candidates for production deployment.


- **Model Governance and Validation Considerations:**  
  From a governance perspective, simpler models such as **Logistic Regression** are the easiest to document, replicate, and explain, requiring limited oversight for validation teams.  
  Tree-based ensembles like **XGBoost**, while more complex, can still be made compliant when paired with structured explainability tools (e.g., SHAP[^4]), version control, and robust monitoring.  
  Given that the chosen model’s interpretability was carefully managed through SHAP and transparent data preprocessing, the additional documentation effort is justified by the **substantial performance gains** and **regulatory explainability** it provides.

In summary, while the simpler models offered transparency and the Random Forest achieved competitive scores, **XGBoost delivered the most balanced, flexible, and explainable predictive performance**—making it the preferred choice for robust credit risk modeling.

---

### Explainability and Fairness Considerations

Model transparency and fairness were also key selection criteria, in line with the **Equal Credit Opportunity Act (ECOA)** [^1] and **Regulation B (12 CFR Part 1002)** [^2].  

The **XGBoost model** was accompanied by **SHAP (SHapley Additive exPlanations)** analysis to assess feature contributions and ensure interpretability.  

SHAP values made the model’s decisions transparent both globally and individually, confirming that the most influential features—such as **debt-to-income ratio**, **past delinquencies**, and **loan value**—were economically sound and not discriminatory.  

This interpretability supports **accountable and fair decision-making**, ensuring no sensitive or proxy variables unduly influence outcomes [^3].

In summary, **XGBoost** was selected not only for its superior predictive performance but also for its **ability to provide transparent, auditable, and economically interpretable explanations**, making it a suitable choice for responsible credit risk modeling.


### Robustness Check 

We estimate the expected costs for each model, assuming that only a % of the loan value could be recovered in case of default and an annualized average interest rate on loans. Under our parametric assumption (40% recovery rate and 4% annualized interest rate), XGBoost model outperforms the other models. However, when the ratio Loss-Given-Default (LGD) over interest rate gets sufficiently small, the Logistic regression model outperforms XGBoost in this dimension. However, this exercise assumes among other things a unique average annualized rate. With more information on the relationship between rate and "riskiness" of the loan, we would be able to refine this measurement and identify with more confidence the best model in terms of cost-effectiveness.


## **2. Problem and Solution Summary** 

The bank’s loan-approval process is manual, slow, and exposed to human judgment errors and biases. Non-performing loans (NPAs) significantly reduce profitability. 

**Objective:** Develop an empirically derived, statistically sound, and interpretable credit-scoring model that predicts the likelihood of default and supports transparent lending decisions.

A data-driven classification model trained on historical applicants was built to predict loan default. Multiple algorithms were benchmarked — Logistic Regression, Decision Tree, Random Forest, AdaBoost, Gradient Boosting, and XGBoost — with rigorous hyperparameter tuning and cross-validation.
XGBoost was selected as the final model due to:
- Highest ROC-AUC and consistent performance across folds (≈0.96).
- Strong recall (68%) ensuring most risky applicants are correctly identified.
- Explainability through SHAP values satisfying compliance and audit requirements.
- Minimal loss of accuracy after removing potentially bias-inducing variables (e.g., JOB).

By implementing the model:
- Loan officers can make faster and more objective credit decisions.
- The bank can reduce default losses by pre-screening high-risk applicants.
- Decision explanations can be generated automatically, enhancing regulatory transparency.
- The overall approval process becomes data-driven and fairer, reducing bias risk and manual workload.


## **3. Recommendations for Implementation** 

### **Key Recommendations:**

1. Integrate the XGBoost model within the loan-application system to generate a default-risk score for every applicant.
2. Use SHAP explanations for each prediction to justify decisions and comply with explainability requirements.
3. Regularly retrain the model (e.g., quarterly) using new loan performance data to maintain accuracy and adapt to changing credit dynamics.
4. Establish monitoring dashboards tracking model performance (AUC, recall, false positives).
5. Continuously monitor the ratio of Loss Given Default (LGD) to the average interest rate to assess whether the XGBoost model remains the most cost-effective choice. Include this metric in regular model-performance dashboards to guide strategic threshold adjustments or temporary fallback to the logistic model when market risk conditions shift. As mentioned in section 1, it is vital that more information on the true costs of false positive and negatives is provided in order to calibrate the metric correctly. 

### **Ethical Risks and Regulation:**

We select XGBoost with SHAP explanations to balance predictive performance and interpretability. SHAP enables per-applicant disclosure of feature-level contributions, thereby facilitating specific reasons in adverse-action notices as required by ECOA / Regulation B even for algorithmic models. (The CFPB has made clear that using a complex model does not excuse non-compliance with adverse-action rules.) Our exclusion of JOB / occupation features is a purposeful bias mitigation step, reducing the risk of embedding socioeconomic proxies. However, omission is not a guarantee of fairness, so we commit to continuous fairness auditing for discriminatory impact or differential error rates. In practice, we will maintain a “less discriminatory alternative” benchmark (logistic regression), monitor classification and cost trade-offs over changing economic conditions (e.g. LGD / interest rate regimes), and provide full model documentation and explanation records for compliance.

These design principles—explainability, bias mitigation, model comparators—help ensure that our advanced model is not just effective, but also defensible under ECOA and fair lending scrutiny.

### **Next Steps/Further Analysis:**

- Add macroeconomic indicators (e.g., unemployment) to improve resilience during downturns.
- Add interest rate information to improve measurement of the model's expected cost. This feature would clearly be endogenous and highly correlated with others so an in-depth analysis/revision of the model is required.
- Calibrate probabilities to ensure that predicted default risks reflect actual observed frequencies. This improves the reliability of risk-based pricing and compliance reporting.
- Incorporate cost-sensitive optimization to align the model’s decision threshold (or training weights) with the asymmetric economic costs of credit errors. This allows the model to directly minimize expected financial loss rather than abstract statistical error. 
- Deploy an A/B pilot comparing manual vs. model-assisted approval performance.
- Run a counterfactual analysis on the XGBoost model to add further interpretability power.
- Implement counterfactual (recourse) analysis to complement SHAP explanations. This would identify the *minimal and actionable changes* an applicant could make to obtain loan approval (e.g., lower debt-to-income ratio, improve credit history). Such counterfactual insights enhance transparency and customer communication while supporting fair-lending objectives under ECOA.


## **4. Conclusion**

In conclusion, the selected XGBoost model, supported by SHAP interpretability and a robust monitoring regime, positions the bank to achieve better risk-adjusted returns while meeting key fairness and compliance requirements.


## **References

[^1] Board of Governors of the Federal Reserve System. *Equal Credit Opportunity Act (Regulation B) Manual* (12 CFR Part 1002). Washington, DC: Federal Reserve Board. [https://www.federalreserve.gov/boarddocs/supmanual/cch/fair_lend_reg_b.pdf](https://www.federalreserve.gov/boarddocs/supmanual/cch/fair_lend_reg_b.pdf)

[^2]: Consumer Financial Protection Bureau (CFPB). *Consumer Financial Protection Circular 2023-03: Adverse action notification requirements and the proper use of the CFPB’s sample forms provided in Regulation B*. [https://www.consumerfinance.gov/compliance/circulars/circular-2023-03-adverse-action-notification-requirements-and-the-proper-use-of-the-cfpbs-sample-forms-provided-in-regulation-b/](https://www.consumerfinance.gov/compliance/circulars/circular-2023-03-adverse-action-notification-requirements-and-the-proper-use-of-the-cfpbs-sample-forms-provided-in-regulation-b/)

[^3]: Ammermann, S. (2013). *Adverse Action Notice Requirements Under the ECOA and the FCRA*. *Consumer Compliance Outlook*, Q2 2013. [https://www.consumercomplianceoutlook.org/2013/second-quarter/adverse-action-notice-requirements-under-ecoa-fcra/](https://www.consumercomplianceoutlook.org/2013/second-quarter/adverse-action-notice-requirements-under-ecoa-fcra/)

[^4]: Lundberg, S.M., & Lee, S.-I. (2017). *A Unified Approach to Interpreting Model Predictions.* *Advances in Neural Information Processing Systems (NeurIPS 2017).* [https://arxiv.org/abs/1705.07874](https://arxiv.org/abs/1705.07874)
